Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2

from dotenv import load_dotenv
from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset


Chargement de la bdd

In [ ]:
dossier_projet = Path.cwd().parent
chemin_env = dossier_projet / ".env"

load_dotenv(chemin_env)

url_base_donnees = os.getenv("DATABASE_URL")

if not url_base_donnees:
    raise ValueError(
        "La variable DATABASE_URL n'est pas définie."
    )

print("Connexion configurée ✅")


Chargement des datas 

In [ ]:
requete = """
SELECT
*
FROM prediction_logs
ORDER BY timestamp;
"""

with psycopg2.connect(url_base_donnees) as connexion:
    donnees_monitoring = pd.read_sql_query(
        requete,
        connexion,
    )

print("Nombre total de lignes :", len(donnees_monitoring))

display(donnees_monitoring.head())


Verif de la répartition

In [ ]:
repartition_groupes = (
    donnees_monitoring["type_donnees"]
    .value_counts(dropna=False)
    .rename_axis("type_donnees")
    .reset_index(name="nombre_lignes")
)

display(repartition_groupes)


Verif des doublons

In [ ]:
identifiants_reference = set(
    donnees_monitoring.loc[
        donnees_monitoring["type_donnees"] == "reference",
        "sk_id_curr",
    ]
)

identifiants_production = set(
    donnees_monitoring.loc[
        donnees_monitoring["type_donnees"] == "production",
        "sk_id_curr",
    ]
)

identifiants_communs = (
    identifiants_reference
    & identifiants_production
)

print(
    "Clients présents dans les deux groupes :",
    len(identifiants_communs),
)


Séparation des datas

In [ ]:
donnees_reference = (
    donnees_monitoring[
        donnees_monitoring["type_donnees"] == "reference"
    ]
    .copy()
    .reset_index(drop=True)
)

donnees_production = (
    donnees_monitoring[
        donnees_monitoring["type_donnees"] == "production"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Référence :", donnees_reference.shape)
print("Production :", donnees_production.shape)


Selection variables de drift

In [ ]:
variables_metier = [
    "amt_credit",
    "nbre_annee",
    "nombre_enfants",
    "anciennete_professionnelle",
    "age",
    "revenu_annuel",
]

variables_prediction = [
    "score_risque",
    "prediction",
]

variables_drift = (
    variables_metier
    + variables_prediction
)

variables_disponibles = [
    colonne
    for colonne in variables_drift
    if colonne in donnees_monitoring.columns
]

print("Variables analysées :")

for colonne in variables_disponibles:
    print("-", colonne)


Prepa des datas

In [ ]:
reference_drift = donnees_reference[
    variables_disponibles
].copy()

production_drift = donnees_production[
    variables_disponibles
].copy()

for colonne in variables_disponibles:
    reference_drift[colonne] = pd.to_numeric(
        reference_drift[colonne],
        errors="coerce",
    )

    production_drift[colonne] = pd.to_numeric(
        production_drift[colonne],
        errors="coerce",
    )

    reference_drift = reference_drift.replace(
    [np.inf, -np.inf],
    np.nan,
)

production_drift = production_drift.replace(
    [np.inf, -np.inf],
    np.nan,
)




Verif valeurs manquantes

In [ ]:
controle_qualite = pd.DataFrame({
    "manquantes_reference": (
        reference_drift.isna().sum()
    ),
    "manquantes_production": (
        production_drift.isna().sum()
    ),
    "valides_reference": (
        reference_drift.notna().sum()
    ),
    "valides_production": (
        production_drift.notna().sum()
    ),
    "uniques_reference": (
        reference_drift.nunique(dropna=True)
    ),
    "uniques_production": (
        production_drift.nunique(dropna=True)
    ),
})

display(controle_qualite)


In [ ]:
statistiques_reference = (
    reference_drift.describe().T[
        ["mean", "std", "min", "50%", "max"]
    ]
    .add_suffix("_reference")
)

statistiques_production = (
    production_drift.describe().T[
        ["mean", "std", "min", "50%", "max"]
    ]
    .add_suffix("_production")
)

comparaison_statistiques = (
    statistiques_reference
    .join(statistiques_production)
)

display(comparaison_statistiques)


Création du rapport

In [ ]:
rapport_drift = Report([
    DataDriftPreset(),
    DataSummaryPreset(),
])

resultat_drift = rapport_drift.run(
    current_data=production_drift,
    reference_data=reference_drift,
)

resultat_drift


Save rapport

In [34]:
dossier_rapports = dossier_projet / "reports"

dossier_rapports.mkdir(
    parents=True,
    exist_ok=True,
)

chemin_rapport = (
    dossier_rapports
    / "rapport_data_drift.html"
)

resultat_drift.save_html(
    str(chemin_rapport)
)

print("Rapport enregistré :", chemin_rapport)


Rapport enregistré : /home/jo/Documents/uv/Projet_8/reports/rapport_data_drift.html


In [ ]:
from scipy.stats import ks_2samp


montants_reference = (
    donnees_reference["amt_credit"]
    .dropna()
)

montants_production = (
    donnees_production["amt_credit"]
    .dropna()
)

statistique_ks, p_value = ks_2samp(
    montants_reference,
    montants_production,
)

print(f"Statistique KS : {statistique_ks:.4f}")
print(f"P-value : {p_value:.4f}")

if p_value < 0.05:
    print("✅ Drift détecté")
else:
    print("❌ Drift non détecté")
